# Lab 11 Challenge: worked solution

The challenge: answer the Helena question with `run_loop`, check that both files were found, and say whether the
second search was a planned split or a dependent hop. This notebook runs on its own: it repeats, unchanged, the
pieces of the lab it needs (the config cell, the index of the 36 files, `retrieve`, the `retrieve` tool and the system
message, `run_loop`, and the word check from Part E), then solves both tasks.

## Setup: the pieces from the lab

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import OpenAI, omit

client = OpenAI()
EMBED_MODEL = settings["EMBEDDING_MODEL"]
MAX_OUTPUT = 200
PRICE_IN = float(settings["PRICE_INPUT_PER_MILLION"])
PRICE_OUT = float(settings["PRICE_OUTPUT_PER_MILLION"])
print("embedding model:", EMBED_MODEL)
print("input:", PRICE_IN, "| output:", PRICE_OUT, "US dollars per million tokens")

embedding model: text-embedding-3-small
input: 0.1 | output: 0.5 US dollars per million tokens


The 36 files, embedded in one request and stored in Chroma in memory, as in Step 1.

In [3]:
import json
import re

import chromadb

CORPUS = ROOT / "data" / "corpus"
files = sorted(CORPUS.glob("*.md"))
texts = [path.read_text() for path in files]
response = client.embeddings.create(model=EMBED_MODEL, input=texts)
vectors = [item.embedding for item in response.data]

chroma = chromadb.Client()
collection = chroma.get_or_create_collection("corpus")
collection.upsert(ids=[path.name for path in files], documents=texts, embeddings=vectors)
print("stored:", collection.count())

stored: 36


`embed` and `retrieve` (Step 2), the tool and `run_retrieve` (Step 4), and the system message.

In [4]:
def embed(text):
    return client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding

In [5]:
def retrieve(query, k):
    got = collection.query(query_embeddings=[embed(query)], n_results=k)
    rows = []
    for i in range(len(got["ids"][0])):
        rows.append({
            "id": got["ids"][0][i],
            "distance": got["distances"][0][i],
            "document": got["documents"][0][i],
        })
    return rows

In [6]:
TOOL_K = 2  # each search returns the 2 nearest files

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "retrieve",
            "description": (
                "Search the shop policy and customer ticket files. "
                "Argument query must be a concise keyword phrase "
                "(e.g. 'gift card expiry', 'student discount', 'Puja ticket'). "
                "Do not repeat keywords from files already retrieved."
            ),
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    }
]

In [7]:
def run_retrieve(query):
    rows = retrieve(query, TOOL_K)
    names = [row["id"] for row in rows]
    content = "\n\n".join(f"{row['id']}\n{row['document']}" for row in rows)
    return names, content

In [8]:
SYSTEM = (
    "You answer questions using only the shop policy and ticket files. Use the retrieve tool to find facts.\n"
    "Each search query should be 2 to 4 keywords for one specific concept (e.g. 'gift card expiry' or 'student discount').\n"
    "If you already retrieved a document, do not repeat its terms in subsequent search queries.\n"
    "If the files do not contain the answer, say you do not know.\n"
    "Do not invent facts."
)

`run_loop` (Step 8), and the word check (Step 9).

In [9]:
def run_loop(question, max_turns=6, verbose=True):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    result = {"searches": [], "calls": 0, "prompt_tokens": 0, "completion_tokens": 0, "largest_prompt": 0,
              "final_text": None, "stopped": "turn cap"}
    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            max_completion_tokens=MAX_OUTPUT,
            reasoning_effort=EFFORT or omit,
        )
        result["calls"] += 1
        result["prompt_tokens"] += response.usage.prompt_tokens
        result["completion_tokens"] += response.usage.completion_tokens
        result["largest_prompt"] = max(result["largest_prompt"], response.usage.prompt_tokens)
        message = response.choices[0].message
        if verbose:
            print("--- turn", turn, "finish_reason:", response.choices[0].finish_reason, "---")
        if not message.tool_calls:
            result["final_text"] = message.content
            result["stopped"] = "answer"
            if verbose:
                print(message.content)
            break
        messages.append(message)
        for call in message.tool_calls:
            query = json.loads(call.function.arguments)["query"]
            names, content = run_retrieve(query)
            result["searches"].append({"turn": turn, "query": query, "files": names})
            if verbose:
                print("retrieve:", query)
                print("  files:", names)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": content})
    if verbose and result["stopped"] == "turn cap":
        print("stopped: cap was", max_turns)
    return result

In [10]:
def words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

In [11]:
def borrowed_words(query, question, files_read):
    text_read = " ".join((CORPUS / name).read_text() for name in files_read)
    return sorted((words(query) - words(question)) & words(text_read))


def label_searches(result, question):
    for search in result["searches"]:
        read_before = [name for s in result["searches"] if s["turn"] < search["turn"] for name in s["files"]]
        borrowed = borrowed_words(search["query"], question, read_before)
        print(f"   turn {search['turn']}: {search['query']!r:48} borrowed: {borrowed}")

## Task 1: Helena's rep and the first-reply time

Before running, the expected answer: ticket 01 (`ticket_01_helena_wrong_album.md`) names her rep, **Steve Johnson**;
the support-hours policy (`policy_support_hours.md`) says the first reply comes **within 1 business day**. Both files
must be in `files_used`.

In [12]:
HELENA = ("Helena received the wrong album. Who is her assigned support representative, "
          "and how soon should support send a first reply?")

out = run_loop(HELENA)
n_retrieves = len(out["searches"])
files_used = [name for search in out["searches"] for name in search["files"]]
final_text = out["final_text"]
searches = out["searches"]
print()
print("n_retrieves:", n_retrieves)
print("files_used: ", files_used)

--- turn 1 finish_reason: tool_calls ---


retrieve: Helena wrong album
  files: ['ticket_01_helena_wrong_album.md', 'ticket_14_duplicate_invoice.md']


--- turn 2 finish_reason: tool_calls ---


retrieve: first reply timing
  files: ['policy_support_hours.md', 'ticket_02_puja_download_expired.md']


--- turn 3 finish_reason: stop ---
Helena’s assigned representative is **Steve Johnson**. Support should send its first reply **within 1 business day**.

n_retrieves: 2
files_used:  ['ticket_01_helena_wrong_album.md', 'ticket_14_duplicate_invoice.md', 'policy_support_hours.md', 'ticket_02_puja_download_expired.md']


Two searches, one per turn: *Helena wrong album* brought ticket 01 (and ticket 14), then *first reply timing* brought `policy_support_hours.md` (and ticket 02). The answer: Steve Johnson, and a first reply within 1 business day. `files_used` holds four names because each search returns two files; the two we need are among them.

The check from the lab, unchanged:

In [13]:
# The check: run it after you have bound the four names in the cell above
if "searches" not in globals():
    print("Bind n_retrieves, files_used, final_text and searches in the cell above first.")
else:
    print("queries:", [(search["turn"], search["query"]) for search in searches])
    assert n_retrieves >= 1, "run_loop should search at least once"
    assert "ticket_01_helena_wrong_album.md" in files_used, "Helena's ticket names her rep"
    assert "policy_support_hours.md" in files_used, "the first-reply time is in the support-hours policy"
    assert "steve" in str(final_text).lower(), "Helena's assigned rep is Steve Johnson"
    assert "1 business day" in str(final_text).lower(), "first reply is within 1 business day"
    print("✅ looks good")

queries: [(1, 'Helena wrong album'), (2, 'first reply timing')]
✅ looks good


The check printed both queries with their turns and passed: both files retrieved, Steve Johnson named, 1 business day stated. It would pass just as well if one search had returned both files, or if both searches had gone out in one reply: it checks what was found, not how.

## Task 2: Planned split or dependent hop?

`label_searches` prints, for each search, the words of its query that are not in the question but are in a file the
agent had already read.

In [14]:
label_searches(out, HELENA)

   turn 1: 'Helena wrong album'                             borrowed: []
   turn 2: 'first reply timing'                             borrowed: []


No query borrows a word from a file. *Helena wrong album* is the question's own words, and *first reply timing* takes "first reply" from the question. Ticket 01 gave the agent an answer, the rep's name, not a pointer to another file. This run is a planned split.

One run is one run, so three more, each labelled:

In [15]:
for run in range(1, 4):
    result = run_loop(HELENA, verbose=False)
    files = [name for search in result["searches"] for name in search["files"]]
    found = "ticket_01_helena_wrong_album.md" in files and "policy_support_hours.md" in files
    text = (result["final_text"] or "").lower()
    print(f"run {run}: searches {len(result['searches'])}, both files {found}, "
          f"Steve {'steve' in text}, 1 business day {'1 business day' in text}")
    label_searches(result, HELENA)

run 1: searches 2, both files True, Steve True, 1 business day True
   turn 1: 'Helena wrong album'                             borrowed: []
   turn 2: 'first response timing'                          borrowed: []


run 2: searches 2, both files True, Steve True, 1 business day True
   turn 1: 'Helena wrong album'                             borrowed: []
   turn 2: 'first reply response time'                      borrowed: []


run 3: searches 2, both files True, Steve True, 1 business day True
   turn 1: 'Helena wrong album'                             borrowed: []
   turn 2: 'support first reply time'                       borrowed: []


The same in all three: two searches, both files, Steve Johnson and 1 business day, and no borrowed word. With the run above, 4 of 4 runs were planned splits.

**What this shows.** The Helena question needs two files, so the agent searches more than once, but it is the same
shape as the "damaged twice" question in Part E: the question itself says what to look for ("Helena", "wrong album",
"first reply"). Ticket 01 answers part of the question (the rep's name); it does not point the agent to another file.
On the ticket-06 question the second search could only be written after reading the ticket. Searching twice is not
the same as hopping.